# CAJAL Computational Mini-Project 16

## Level 1 — Exploring neuronal diversity programming from large-scale combinatorial morphogen perturbation screen

---

You have been given a single-cell RNA-seq dataset from a large-scale combinatorial morphogen perturbation screen. The dataset was generated using Parse single cell technology and there are 2 plates and 192 perturbations in total. You will first need to analyse the data and figure out what kind of neuronal cells are generated.

**Important:** Do not look up the source paper for this dataset.

**Remember:** Use `function_name?` or `help(function_name)` when you encounter something unfamiliar. The [scanpy API docs](https://scanpy.readthedocs.io/en/stable/api/index.html) and the [Single-cell best practices book](https://www.sc-best-practices.org/) are your main references. Additionally, the [Orchestrating Single-Cell Analysis with Bioconductor book](https://bioconductor.org/books/release/OSCA/) is a recommended supplementary reference, though this book is designed for running analysis primarily in R.


In [ ]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

# Required when obs/var contain pandas StringArray columns.
ad.settings.allow_write_nullable_strings = True
pd.options.future.infer_string = False

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(5, 4))

%matplotlib inline

In [ ]:
adata = sc.read_h5ad("data/ngn2_scrna_raw.h5ad")
print(adata)

## Part 1: Quality Control


Before any further analysis, it is essential to remove low-quality cells. Cells with very few detected genes or very low total counts are likely empty droplets (in droplet based sequencing technology such as 10x) or debris in our case. Conversely, cells with unusually high counts may be doublets (two cells captured as one). A high fraction of mitochondrial reads is a hallmark of dying or damaged cells, whose cytoplasmic mRNA has leaked out while mitochondrial RNA remains. Similarly, ribosomal gene content can flag stressed or aberrant cells.

Calculate per-cell QC metrics - total counts, genes detected, % mitochondrial and ribosomal counts - using the scanpy helper function. Visualise the distributions per plate and use these plots to justify your filtering thresholds. You can apply a fixed threshold for filtering or alternatively, use a MAD (median absolute deviation) based outlier strategy. You should also consider if you will need to perform the threshold detection on each plate separately. You may also optionally filter genes detected in very few cells - though this is primarily to reduce the computational complexity.

## Part 2: Normalisation and Feature Selection


Raw count data cannot be compared directly across cells because sequencing depth varies between cells — a gene appearing to be expressed more in one cell may simply reflect more total reads. Count normalisation scales the counts in each cell to remove effect of different sequncing variation. One standard approach in scanpy is library-size normalisation, followed by log-transform. Before normalising, save the raw counts as some downstream method require raw counts.

Following normalisation, we will perform feature selection. Most genes show little to no variation across cells - they are either housekeeping genes which is always expressed or always off. We want to identify the most informative genes - highly variable genes (HVGs) - that vary meaningfully across cells rather than just due to noise. Downstream steps will operate on this reduced gene set.

## Part 3: Dimensionality Reduction and Integration


After HVGs gene selection, we are still left with thousands of genes, which is too high-dimensional to visualise or cluster directly. One technique for reducing high-dimensional data to low-dimensional representation is **Principal Component Analysis**, which finds the linear combinations of genes that capture the most variance. PC1 is the direction of maximum variance, PC2 is the next (orthogonal to PC1), and so on. Typically, you would use 30-50 PCs, though given that this dataset may produce large number of cell types, you may need to use higher number of PCs.  Plot the variance explained per PC to find the "elbow" — the point where adding more PCs gives diminishing returns.

This dataset was collected across two sequencing plates, which introduce technical batch effects - systematic differences between plates which are unrelated to biology. If uncorrected, cells may cluster by plate rather than by cell type. You can use batch correction techniques, such as **Harmony** or **scVI** to correct for technical batch effects. We shall use Harmony as it is built into scanpy. Harmony works by operating on the PCA embedding: it iteratively nudges each cell's coordinates so that the same cell type from different plates overlaps, while preserving genuine biological structure.

## Part 4: Visualisation

UMAP provides a 2-D visualisation of the high-dimensional transcriptome by first building a nearest-neighbour graph in PCA space and then optimising a low-dimensional layout. Here you will compute UMAPs both with and without Harmony correction, so you can directly compare the effect of batch integration. Use the corrected embedding for all downstream analyses.


## Part 5: Clustering

Leiden clustering partitions the neighbour graph into communities of transcriptionally similar cells. The resolution parameter controls granularity — too low and distinct populations merge; too high and single populations fragment artificially.

Run clustering across a range of resolutions and visualise each result on the UMAP. Choose a resolution that captures biologically meaningful distinctions and assign it as your working clustering. Typical values range from 0.3 to 1.0, though given the large number of cell types produced by the screen, you may need to use higher values such as 10.0.

## Part 6: Cell Type Annotation


Now that cells are clustered, the goal is to assign biological identities to each cluster. This is the most important part of the analysis — and the most creative.

A typical first step in cell type annotation is to find **marker genes** - genes that are specifically expressed in each cluster versus the rest. This works well particularly if you have distinct cell types in your data. You may also consider using known cell type signatures to help with cell type annotation, e.g. through visualising marker genes with dotplot or using gene score method to score cells for sets of cell type signatures.

Given the large number of neuronal cell types generated by the screen, we will focus less on annotating each cluster to a given cell type. Rather, we are more interested in the types of neuron being created. Begin with a manual inspection — plot canonical neuronal marker genes (`MAP2`, `PRPH`, `SLC17A6`, `CHAT`, `SLC6A2`, `GAD1`, `MKI67`) on the UMAP to orient yourself.

The `AP_axis` and `DV_axis` metadata columns encode which anterior-posterior (AP) and dorso-ventral (DV) axis transcription factors combination was used for each cell (with numeric replicate suffixes). Consider simplifying these columns and overlaying the condition on the UMAP to understand how the programming strategy relates to transcriptional identity.

Aside from manually annotating each cell type, another option for cell type annotation is using automated cell type annotation methods, such as by projecting your data into a reference/atlas dataset as implemented by tools such as **scANVI** or through machine learning/classifier method as implemented by tools such as **CellTypist**. You would normally have to collect reference neuronal datasets, which spans various neuron type and region.

## Part 7: Making Figures

Throughout your analysis, you have been generating exploratory plots to check distributions, testing parameters, examining gene expression and more. These plots exist in notebooks and seen only by you, so speed and efficiency matter more than polish.

When you want to publish your analysis in papers, presentation or posters, you will need to create publication-quality figures. A good publication figure is self-explanatory — a reader should understand what it shows without consulting the methods section. This demands careful attention to font sizes (axes and labels should be readable when printed at 10–12pt; Scanpy defaults are often too small), axis labels with units, informative titles, and consistent color schemes. For multi-condition comparisons, use subplots with consistent axis ranges and a single shared legend so the reader can easily compare across panels.

Review your analysis and select 2–3 of your most informative exploratory plots to remake as publication-quality figures. Ensure colors are distinguishable and, where possible, colorblind-friendly and consistent across related panels. All axes must be labeled, and multi-panel figures need titles or letters (a, b, c) to guide the reader. Legends must be complete and informative. Finally, export your figures in vector format (i.e. pdf) for a publication-ready output.

## Part 8: Save Results and Summary


Save the fully processed and annotated AnnData object for use in downstream analyses.
